In [1]:
from __future__ import print_function

import argparse
import json

from deep_disfluency.asr.ibm_watson import IBMWatsonAdapter
from deep_disfluency.tagger.deep_tagger_module import DeepTaggerModule
import pathlib2 as pathlib
from six.moves import queue


def queue_consumer(queue):
    while not queue.empty():
        yield queue.get()


def dd(in_data, tagger):
    '''
    Run DD for a list of Watson input data. Produce a list of output data.
    '''
    # Prepare adapter
    adapter = IBMWatsonAdapter()
    # Necessary because of fluteline
    adapter.enter()
    adapter.output = queue.Queue()

    # Prepare tagger
    # Necessary because of fluteline
    tagger.output = queue.Queue()
    # Necessary because fluteline's enter is not implemented
    tagger.disf_tagger.reset()
    tagger.latest_word_ID = -1
    tagger.word_graph = []

    for item in in_data:
        # if item['results'][0]['final']:
        adapter.consume(item)

    for item in queue_consumer(adapter.output):
        tagger.consume(item)

    return list(tagger.word_graph)



def main():
    args_in_dir = "fake_asr_results"
    args_out_dir = "fake_dd_results"

    in_dir = pathlib.Path(args_in_dir)
    assert in_dir.is_dir(), 'No such directory: {}'.format(in_dir)

    out_dir = pathlib.Path(args_out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    tagger = DeepTaggerModule()

    for in_filepath in sorted(in_dir.iterdir()):
        print('Processing {}'.format(in_filepath))

        with in_filepath.open() as f:
            in_data = json.load(f)

        in_data = [x for x in in_data if x['results'][0]['final']]

        out_data = dd(in_data, tagger)

        out_filepath = (out_dir / in_filepath.name).with_suffix('.json')
        with out_filepath.open('w', encoding='utf-8') as f:
            # Stupid python 2: https://stackoverflow.com/q/36003023/1224456
            f.write(unicode(json.dumps(out_data)))


In [2]:
main()

Initializing Tagger
Processing args from config number 35 ...
Intializing model from args...
Using the cpu
	Adjust Theano config file ($HOME/.theanorc)
loading tag to index maps...
Initializing model of type lstm ...


/anaconda2/lib/python2.7/site-packages/deep_disfluency/rnn/lstm.py:112: UserWarning: DEPRECATION: If x is a vector, Softmax will not automatically pad x anymore in next releases. If you need it, please do it manually. The vector case is gonna be supported soon and the output will be a vector.
  y_t = T.nnet.softmax(T.dot(h_t, self.W_hy) + self.b_y)


Loading saved weights from /anaconda2/lib/python2.7/site-packages/deep_disfluency/tagger/../experiments/035/epoch_6
No POS tagger specified,loading default CRF switchboard one
No timer specified, using default switchboard one
Loading decoder...
loading swbd_disf1_uttseg_simple_033 Markov model
Markov Model ready mode:
constraint only
Deep Tagger Module ready
Processing fake_asr_results/offline_watson.json


/anaconda2/lib/python2.7/site-packages/sklearn/base.py:253: UserWarning: Trying to unpickle estimator LogisticRegression from version 0.18.1 when using version 0.20.3. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
/anaconda2/lib/python2.7/site-packages/sklearn/base.py:253: UserWarning: Trying to unpickle estimator StandardScaler from version 0.18.1 when using version 0.20.3. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
